In [1]:
#%% PACKAGES
# Basics
import numpy as np                                                          
from math import pi                                                            
import warnings                                                              
import os    


from datetime import datetime, timedelta
from pandas import DataFrame
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler

# Visualization
import pandas as pd                                                          
import matplotlib.pyplot as plt                                                
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import numpy as np
import seaborn as sns; sns.set_theme(style='white')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
IndexRaw = gpd.read_parquet(f'{output_step2_path}/step2_features.parquet')

attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")

In [3]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,attribute_source_path,initial_weight,class_weight,impact_attribut,file_name,geometry_type,method,how,value_column,buffer_size,filter_column,filter_values,crs,save_format
0,Agrément,attente,attente,False,attente,0.3,0.3,defavorable,NaN,point,A,sum,completer,10,filtered,1,2056,parquet
1,Agrément,bruit,bruit,True,bruit,0.4,0.4,defavorable,SPBR_SECTEUR_EXPOSE_AU_BRUIT_2025/SPBR_SECTEUR...,polygon,A,area_ratio,NaN,30,filtered,1,2056,parquet
2,Agrément,temperature,temperature,True,temperature,0.6,0.6,defavorable,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,point,A,raster,temperature,10,filtered,1,2056,parquet
3,Agrément,vegetation,canopee,True,canopee,0.5,0.5,favorable,SIPV_ICA_MNC_2023-SHP/SIPV_ICA_MNC_2023_QGISsi...,polygon,A,length_area_ratio,NaN,30,filtered,1,2056,parquet
4,Agrément,vegetation,arbre_isole,False,arbre_isole,NaN,NaN,favorable,SIPV_ICA_ARBRE_ISOLE-SHP/SIPV_ICA_ARBRE_ISOLE.shp,point,A,sum,D_COURONNE,20,filtered,1,2056,parquet
5,Agrément,vegetation,espace_vert,False,domaine_routier,NaN,NaN,favorable,CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
6,Attractivité,eau,eau,True,eau,0.3,0.3,favorable,LCE_GRAPHE_EAU-SHP/LCE_GRAPHE_EAU.shp,line,A,count,NaN,10,filtered,1,2056,parquet
7,Attractivité,espaces_ouverts,espaces_ouverts,True,espaces_ouverts,0.8,0.8,favorable,OBS_EQUIPEMENTS_ESPACES_PUB-SHP/OBS_EQUIPEMENT...,polygon,A,count,NaN,10,filtered,1,2056,parquet
8,Attractivité,proximite,rez_actif,True,rez_actif,0.5,0.5,favorable,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,count,NaN,10,filtered,1,2056,parquet
9,Attractivité,proximite,tp,True,tp,0.6,0.6,favorable,TPG_ARRETS-SHP/TPG_ARRETS.shp,point,A,count,NaN,200,filtered,1,2056,parquet


In [4]:
print(attributs_info['attribute'].to_list())

['attente', 'bruit', 'temperature', 'canopee', 'arbre_isole', 'espace_vert', 'eau', 'espaces_ouverts', 'rez_actif', 'tp', 'amenite', 'connectivite', 'largeur_trottoir', 'stationnement_genant', 'topographie', 'accident', 'charge', 'zone_apaisee', 'zone_pietonne', 'vitesse', 'toilette', 'ratio_trottoir', 'objets_divers']


In [5]:
#Delete where includes in index = False
attributs_info = attributs_info[attributs_info['include_in_index'] != False]
# Fill NULL
Indexv1 = IndexRaw.fillna(0)

# Create mapping for processed columns while keeping first 5 columns unchanged
attribute_mapping = {}
for _, row in attributs_info.iterrows():
    old_name = f"{row['attribute']}_{row['method']}_{row['buffer_size']}"
    new_name = row['attribute']
    attribute_mapping[old_name] = new_name

# Keep first 5 columns as is, rename the rest using the mapping
first_5_cols = IndexRaw.columns[:5].tolist()
Indexv1 = Indexv1.rename(columns=attribute_mapping)

# Debug info
print("First 5 columns:", first_5_cols)
print("Renamed columns:", [col for col in Indexv1.columns if col not in first_5_cols])

print('Crop outliers if necessary: --> see Walkability Amsterdam notebook')
Indexv2 = Indexv1.copy()

#Adding intervals (for factors where we have an interval of interest and below or above that interval the situation doesn't affect the walkability)
#Indexv2['stationnement_genant'] = Indexv1['stationnement_genant'].clip(upper=5) #more than 20 stationnement_genant in 10-meters-radius around segment centroid

# Create copy and normalize
Indexv3 = Indexv2.copy()
scaler = MinMaxScaler()

# Track changes during normalization
print("Min max normalization...")
for attribute in attributs_info['attribute']:
    if attribute in Indexv3.columns:
        # Normalize
        Indexv3.loc[:, attribute] = scaler.fit_transform(Indexv3[[attribute]]).round(4)
    else:
        print(f"Attribute '{attribute}' not found in Indexv3 columns.")
# Inverse columns (for factors that have a negative effect on walkability) depending on attributs_info.impact_attribut (favorable or defavorable)
print("Inverse columns where necessary...")
for _, row in attributs_info.iterrows():
    attribute_name = row['attribute']
    impact = row['impact_attribut']
    if attribute_name in Indexv3.columns:
        if impact == 'defavorable':
            Indexv3[attribute_name] = 1 - Indexv3[attribute_name]
            print(f"Inverted attribute: {attribute_name}")
    else:
        print(f"Attribute '{attribute_name}' not found in Indexv3 columns.")


First 5 columns: ['geometry', 'segment_id', 'length', 'bruit_A_30', 'temperature_A_10']
Renamed columns: ['bruit', 'temperature', 'canopee', 'eau', 'espaces_ouverts', 'rez_actif', 'tp', 'amenite', 'connectivite', 'largeur_trottoir', 'stationnement_genant', 'topographie', 'accident', 'zone_apaisee', 'zone_pietonne', 'vitesse']
Crop outliers if necessary: --> see Walkability Amsterdam notebook
Min max normalization...
Inverse columns where necessary...
Inverted attribute: bruit
Inverted attribute: temperature
Inverted attribute: stationnement_genant
Inverted attribute: topographie
Inverted attribute: accident
Inverted attribute: vitesse


/var/folders/xd/q3z8hkl97ss7mdr3m6y7rn340000gn/T/ipykernel_2299/3982586524.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.0034 0.0535 0.2043 ... 0.0899 0.0129 0.0042]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  Indexv3.loc[:, attribute] = scaler.fit_transform(Indexv3[[attribute]]).round(4)


In [6]:
Indexv3

,geometry,segment_id,length,bruit,temperature,canopee,eau,espaces_ouverts,rez_actif,tp,amenite,connectivite,largeur_trottoir,stationnement_genant,topographie,accident,zone_apaisee,zone_pietonne,vitesse
0,"LINESTRING (6.22016 46.20288, 6.22023 46.20272)",000000,18.611510,0.8945,0.0683,1.0000,0.0,0.0000,0.0000,0.1579,0.0000,0.0034,0.0870,1.0000,0.9944,1.0000,0.0,0.0000,0.8331
1,"LINESTRING (6.20635 46.19641, 6.20656 46.19634)",000001,17.373877,0.5556,0.0350,0.0615,0.0,0.1111,0.0000,0.2632,0.0000,0.0535,0.1739,1.0000,0.9900,0.9714,0.0,0.0000,0.8427
2,"LINESTRING (6.14334 46.20439, 6.14354 46.2041,...",000002,50.000000,0.5942,0.0382,0.0000,0.0,0.4444,0.0222,0.4737,0.0222,0.2043,0.3913,0.9938,0.8874,0.7714,0.0,0.0125,0.8432
3,"LINESTRING (6.1436 46.20398, 6.14362 46.20391)",000003,7.198671,0.6223,1.0000,0.0000,0.0,0.2222,0.0000,0.4211,0.0000,0.1351,0.3043,0.9990,0.9608,0.8571,0.0,0.0115,0.8424
4,"LINESTRING (6.12495 46.18674, 6.12437 46.18679)",000004,45.125485,0.7508,0.0473,0.2966,0.0,0.2222,0.0111,0.2105,0.0111,0.1317,0.4348,1.0000,0.9112,1.0000,0.0,0.1039,0.9168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134065,"LINESTRING (6.1299 46.16566, 6.12997 46.16563)",134065,6.136427,0.9515,1.0000,0.0135,0.0,0.0000,0.0000,0.0000,0.0000,0.0110,0.0000,1.0000,0.9851,1.0000,0.0,0.0000,0.8331
134066,"LINESTRING (6.11294 46.19517, 6.11291 46.19526)",134066,9.836936,0.5556,1.0000,0.0000,0.0,0.0000,0.0000,0.2105,0.0000,0.0248,0.0000,1.0000,0.9572,1.0000,0.0,0.0000,0.8331
134067,"LINESTRING (6.13432 46.18385, 6.13435 46.18382...",134067,9.957378,0.8889,1.0000,0.5397,0.0,0.0000,0.0000,0.3158,0.0000,0.0899,0.0000,1.0000,0.9873,1.0000,0.0,0.0000,0.8331
134068,"LINESTRING (6.10253 46.21532, 6.10261 46.21526)",134068,8.728223,0.8889,0.0309,0.1640,0.0,0.0000,0.0000,0.1053,0.0000,0.0129,0.0000,1.0000,0.9989,1.0000,0.0,0.0000,0.8331


In [7]:
# Calculate the Main Index Scores
Indexv4 = Indexv3

# Get weights from attributs_info and check validity
Zscore_weights = {}
for _, row in attributs_info[attributs_info.include_in_index == True].iterrows():
    attr = row['attribute']
    weight = row['initial_weight']
    if pd.isnull(weight):
        raise ValueError(f"Weight not specified for attribute '{attr}' (found NaN). Please specify a value between 0 and 1.")
    if not (0 <= weight <= 1):
        raise ValueError(f"Weight for attribute '{attr}' is {weight}, but must be between 0 and 1.")
    Zscore_weights[attr] = weight

print("Zscore_weights:", Zscore_weights)


# add zscore weights to the attributes_info dataframe
attributs_info['weights'] = attributs_info['attribute'].map(Zscore_weights)

# Function Sum-product to calculate sub-index score
def calculate_index(df, weights_dict):
    return sum(df[col] * weight for col, weight in weights_dict.items())

Indexv4['I-Zscore'] = calculate_index(Indexv4, Zscore_weights)
Indexv4['I-Zscore_unweighted'] = calculate_index(Indexv4, {key: 1 for key in Zscore_weights.keys()})

#Normalising Index
#Indexv4['Non-Scaled']=Indexv4['I-Zscore'].round(4)
Indexv4[['I-Zscore']] = scaler.fit_transform(Indexv4[['I-Zscore']]).round(4)
Indexv4[['I-Zscore_unweighted']] = scaler.fit_transform(Indexv4[['I-Zscore_unweighted']]).round(4)


# Attribut zone piétonne est prédominant
Indexv5 = Indexv4
max_score = Indexv4["I-Zscore"].max()

Indexv5["walkability_index"] = np.where(
    Indexv5["zone_pietonne"] > 0,
    max_score,               # assign max score if pedestrian zone
    Indexv5["I-Zscore"]           # else keep original
)


Zscore_weights: {'bruit': 0.4, 'temperature': 0.6, 'canopee': 0.5, 'eau': 0.3, 'espaces_ouverts': 0.8, 'rez_actif': 0.5, 'tp': 0.6, 'amenite': 0.6, 'connectivite': 0.7, 'largeur_trottoir': 0.5, 'stationnement_genant': 0.3, 'topographie': 0.4, 'accident': 0.3, 'zone_apaisee': 0.5, 'zone_pietonne': 1.0, 'vitesse': 0.6}


/var/folders/xd/q3z8hkl97ss7mdr3m6y7rn340000gn/T/ipykernel_2299/3724793638.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attributs_info['weights'] = attributs_info['attribute'].map(Zscore_weights)


In [8]:
# Calculate index for each class
Indexv6 = Indexv5


for cls in attributs_info['Class'].unique():
    # Select attributes belonging to this class and included in the index
    subset = attributs_info[
        (attributs_info['Class'] == cls) & 
        (attributs_info['include_in_index'])
    ]

    # Build the weight dictionary for this class only
    cls_weights = {row['attribute']: row['class_weight'] for _, row in subset.iterrows()}

    # Compute weighted sub-index
    Indexv6[f"I-Zscore_{cls}"] = calculate_index(Indexv6, cls_weights)
    # Normalize the result
    Indexv6[[f"I-Zscore_{cls}"]] = scaler.fit_transform(Indexv6[[f"I-Zscore_{cls}"]]).round(4)

In [9]:
Indexv6.head()

,geometry,segment_id,length,bruit,temperature,canopee,eau,espaces_ouverts,rez_actif,tp,amenite,connectivite,largeur_trottoir,stationnement_genant,topographie,accident,zone_apaisee,zone_pietonne,vitesse,I-Zscore,I-Zscore_unweighted,walkability_index,I-Zscore_Agrément,I-Zscore_Attractivité,I-Zscore_Infrastructure,I-Zscore_Sécurité
0,"LINESTRING (6.22016 46.20288, 6.22023 46.20272)",000000,18.611510,0.8945,0.0683,1.0000,0.0,0.0000,0.0000,0.1579,0.0000,0.0034,0.0870,1.0000,0.9944,1.0000,0.0,0.0000,0.8331,0.3898,0.4957,0.3898,0.5975,0.0773,0.3459,0.3212
1,"LINESTRING (6.20635 46.19641, 6.20656 46.19634)",000001,17.373877,0.5556,0.0350,0.0615,0.0,0.1111,0.0000,0.2632,0.0000,0.0535,0.1739,1.0000,0.9900,0.9714,0.0,0.0000,0.8427,0.2553,0.3047,0.2553,0.1793,0.2013,0.4271,0.3194
2,"LINESTRING (6.14334 46.20439, 6.14354 46.2041,...",000002,50.000000,0.5942,0.0382,0.0000,0.0,0.4444,0.0222,0.4737,0.0222,0.2043,0.3913,0.9938,0.8874,0.7714,0.0,0.0125,0.8432,0.4334,0.4295,1.0000,0.1703,0.5416,0.6084,0.2892
3,"LINESTRING (6.1436 46.20398, 6.14362 46.20391)",000003,7.198671,0.6223,1.0000,0.0000,0.0,0.2222,0.0000,0.4211,0.0000,0.1351,0.3043,0.9990,0.9608,0.8571,0.0,0.0115,0.8424,0.5406,0.5614,1.0000,0.5642,0.3510,0.5439,0.3047
4,"LINESTRING (6.12495 46.18674, 6.12437 46.18679)",000004,45.125485,0.7508,0.0473,0.2966,0.0,0.2222,0.0111,0.2105,0.0111,0.1317,0.4348,1.0000,0.9112,1.0000,0.0,0.1039,0.9168,0.4519,0.4975,1.0000,0.3152,0.2579,0.5897,0.4198


In [10]:
# Save it
Indexv6.to_crs(target_crs).to_csv(f'{output_step3_path}/step3_index.csv', index = False)
Indexv6.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_index.parquet')
Indexv6.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_index.gpkg"), driver="GPKG")